# Kepler's laws of planetary motion

*astro · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/kepler_orbits)

Planets with different orbit sizes and eccentricities circle a star under Newton's inverse-square gravity. One batched RK4 integration shows all three of Kepler's laws: ellipses with the star at a focus, equal areas in equal times, and $T^2 \propto a^3$.

**Vector form**

$$\ddot{\mathbf r} = -\frac{GM}{r^3}\,\mathbf r,\qquad \frac{dA}{dt} = \tfrac12\,\lvert\mathbf r\times\dot{\mathbf r}\rvert,\qquad T^2 = \frac{4\pi^2}{GM}\,a^3$$

**Index (tensor) form**

$$\ddot x_{bi} = -GM\,(x_{bj}x_{bj})^{-3/2}\,x_{bi},\qquad L_b = \varepsilon_{ij}\,x_{bi}\,v_{bj}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `y[:, 0, :]` | (B, 2) | position of planet $b$ relative to the star, $i \in \{x, y\}$ |
| $v_{bi}$ | `y[:, 1, :]` | (B, 2) | velocity of planet $b$ |
| $x_{bj}x_{bj}$ | `einsum('bj,bj->b', x, x)` | (B,) | repeated $j$ ⇒ summed: squared distance $r_b^2$ |
| $\ddot x_{bi}$ | `acceleration(x, GM)` | (B, 2) | inverse-square pull toward the star |
| $\varepsilon_{ij}$ | `EPS2` | (2, 2) | 2D Levi-Civita symbol, $\varepsilon_{xy} = 1 = -\varepsilon_{yx}$ |
| $L_b$ | `einsum('ij,bi,bj->b', EPS2, x, v)` | (B,) | angular momentum per unit mass; $dA/dt = L_b/2$ is constant |
| $E_b$ | `energy(y, GM)` | (B,) | $\tfrac12 v^2 - GM/r$; gives $a = -GM/2E$ |

Units are AU, years and solar masses, so $GM_\odot = 4\pi^2$ and Kepler's third law reads simply $T^2 = a^3$. Every planet starts at perihelion with the vis-viva speed $v_p = \sqrt{(GM/a)(1+e)/(1-e)}$. The shaded slices of the most eccentric orbit each take the same time $T/12$: short and wide near the star, long and thin far away, but always the same area, because $dA/dt = L/2$ and $L$ is conserved. The period is measured from the simulation as the time to sweep $2\pi$. The star is treated as fixed (its mass is much larger than the planets') and the planets do not pull on each other.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/kepler_orbits/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    GM = tensors.GM_SUN
    a, e = tensors.orbit_shapes(params["n_planets"], params["e_max"])
    y0 = tensors.initial_state(a, e, GM, dev)

    times, traj = solve.solve(y0, GM, params["t_max"], params["dt"])
    periods = solve.measured_periods(times, traj)

    E, L = equation.energy(traj, GM), equation.angular_momentum(traj)  # (T, B) each
    for b in range(len(a)):
        print(
            f"a = {a[b]:.2f} AU  e = {e[b]:.2f}  T = {periods[b]:.3f} yr  T²/a³ = {periods[b] ** 2 / a[b] ** 3:.4f}"
            f"  ΔE/E = {(E[:, b] / E[0, b] - 1).abs().max():.1e}  ΔL/L = {(L[:, b] / L[0, b] - 1).abs().max():.1e}"
        )
    return plot.animate(times, traj, a, e, periods, out_dir / "kepler_orbits.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/kepler_orbits/equation.py
r"""Physics → tensor translation.

Newton's law of gravity for a planet around a star of mass M (the star is much heavier, so it stays put):

    \ddot{\mathbf r} = -\frac{GM}{r^3}\,\mathbf r

In index notation for a batch of planets b and components i ∈ {x, y}:

    \ddot x_{bi} = -GM \,(x_{bj} x_{bj})^{-3/2}\, x_{bi}

A repeated index (j above) is summed — that is einsum('bj,bj->b'). Two quantities stay constant:

    energy per unit mass            E_b = ½ v_{bi} v_{bi} - GM / \sqrt{x_{bj} x_{bj}}
    angular momentum per unit mass  L_b = \varepsilon_{ij} x_{bi} v_{bj}          (i and j summed)

L_b / 2 is the area swept per unit time (Kepler's second law), and the semi-major axis a_b = -GM / (2 E_b)
fixes the period T_b = 2π \sqrt{a_b^3 / GM} (Kepler's third law).
"""

import torch

EPS2 = torch.tensor([[0.0, 1.0], [-1.0, 0.0]])  # ε_ij, the 2D Levi-Civita symbol


def acceleration(x: torch.Tensor, GM: float) -> torch.Tensor:
    r"""\ddot x_{bi} = -GM (x_{bj} x_{bj})^{-3/2} x_{bi}.   x: (B, 2) → (B, 2)"""
    r2 = torch.einsum("bj,bj->b", x, x)  # (B,)  squared distance to the star
    return -GM * torch.einsum("b,bi->bi", r2**-1.5, x)


def rhs(y: torch.Tensor, GM: float) -> torch.Tensor:
    """dy/dt for the state y of shape (B, 2, 2): y[:, 0] = position, y[:, 1] = velocity."""
    return torch.stack([y[:, 1], acceleration(y[:, 0], GM)], dim=1)


def energy(y: torch.Tensor, GM: float) -> torch.Tensor:
    r"""E_b = ½ v_{bi} v_{bi} - GM / r_b.   y: (..., 2, 2) → (...)"""
    x, v = y[..., 0, :], y[..., 1, :]
    return 0.5 * torch.einsum("...i,...i->...", v, v) - GM / torch.einsum("...i,...i->...", x, x).sqrt()


def angular_momentum(y: torch.Tensor) -> torch.Tensor:
    r"""L_b = ε_{ij} x_{bi} v_{bj} (= x v_y - y v_x).   y: (..., 2, 2) → (...)"""
    x, v = y[..., 0, :], y[..., 1, :]
    return torch.einsum("ij,...i,...j->...", EPS2.to(y), x, v)


def areal_velocity(y: torch.Tensor) -> torch.Tensor:
    """dA/dt = L / 2: the area swept per unit time by the line star → planet."""
    return 0.5 * angular_momentum(y)


def semi_major_axis(y: torch.Tensor, GM: float) -> torch.Tensor:
    """a = -GM / (2E) for a bound orbit."""
    return -GM / (2 * energy(y, GM))


def kepler_period(a: torch.Tensor, GM: float) -> torch.Tensor:
    r"""Kepler's third law T = 2π \sqrt{a^3 / GM}."""
    return 2 * torch.pi * torch.sqrt(a**3 / GM)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/kepler_orbits/tensors.py
"""Build the concrete tensors: orbit sizes, eccentricities and the batched initial state.

Units: astronomical units (AU), years and solar masses, so GM_sun = 4π² AU³/yr² and an orbit with
a = 1 AU takes exactly one year.
"""

import math

import torch

GM_SUN = 4 * math.pi**2  # AU³ / yr²


def orbit_shapes(n: int, e_max: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Semi-major axes a_b (AU) and eccentricities e_b, both (B,). Wider orbits are more eccentric."""
    a = torch.linspace(0.6, 2.2, n) if n > 1 else torch.tensor([1.0])
    e = torch.linspace(0.05, e_max, n) if n > 1 else torch.tensor([e_max])
    return a, e


def initial_state(a: torch.Tensor, e: torch.Tensor, GM: float, device: torch.device) -> torch.Tensor:
    r"""y0 of shape (B, 2, 2): every planet starts at perihelion on the +x axis, moving in +y.

    perihelion distance  r_p = a (1 - e)
    perihelion speed     v_p = \sqrt{(GM / a)(1 + e) / (1 - e)}     (vis-viva equation)
    """
    a, e = a.to(device), e.to(device)
    r_p = a * (1 - e)
    v_p = torch.sqrt(GM / a * (1 + e) / (1 - e))
    zero = torch.zeros_like(a)
    position = torch.stack([r_p, zero], dim=-1)  # (B, 2)
    velocity = torch.stack([zero, v_p], dim=-1)  # (B, 2)
    return torch.stack([position, velocity], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/kepler_orbits/solve.py
"""Integrate all planets at once with RK4, then read the period of each orbit off the trajectory."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(y0: torch.Tensor, GM: float, t_max: float, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2, 2)). One RK4 step moves every planet."""
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, GM), y0, dt, n_steps)
    times = torch.arange(n_steps + 1, device=y0.device, dtype=y0.dtype) * dt
    return times, traj


def measured_periods(times: torch.Tensor, traj: torch.Tensor) -> torch.Tensor:
    """Time at which each planet's polar angle first reaches 2π (one full orbit), (B,). NaN if never."""
    x, y = traj[:, :, 0, 0], traj[:, :, 0, 1]  # (T, B)
    angle = torch.atan2(y, x)  # (T, B) in (-π, π]
    step = torch.remainder(angle.diff(dim=0) + torch.pi, 2 * torch.pi) - torch.pi  # unwrap jumps
    swept = torch.cat([torch.zeros_like(angle[:1]), step.cumsum(0)])  # (T, B) total angle swept
    periods = torch.full((traj.shape[1],), float("nan"), dtype=times.dtype)
    for b in range(traj.shape[1]):
        k = int(torch.searchsorted(swept[:, b].contiguous(), torch.tensor(2 * torch.pi, dtype=swept.dtype)))
        if 0 < k < len(times):  # linear interpolation between steps k-1 and k
            f = (2 * torch.pi - swept[k - 1, b]) / (swept[k, b] - swept[k - 1, b])
            periods[b] = times[k - 1] + f * (times[k] - times[k - 1])
    return periods


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/kepler_orbits/plot.py
"""Animate the orbits with equal-time sectors (2nd law) next to a T² vs a³ plot (3rd law)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Polygon

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    times: torch.Tensor,
    traj: torch.Tensor,
    a: torch.Tensor,
    e: torch.Tensor,
    periods: torch.Tensor,
    path: Path,
    n_slices: int = 12,
) -> Path:
    t = times.cpu().numpy()  # (T,)
    pos = traj[:, :, 0, :].cpu().numpy()  # (T, B, 2)
    a, e, periods = a.cpu().numpy(), e.cpu().numpy(), periods.cpu().numpy()
    B = pos.shape[1]
    hb = B - 1  # highlight the most eccentric planet with equal-time sectors

    fig, (ax, ax_k) = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1.35, 1]})
    lo, hi = pos.reshape(-1, 2).min(0), pos.reshape(-1, 2).max(0)
    pad = 0.08 * (hi - lo).max()
    ax.set_xlim(lo[0] - pad, hi[0] + pad)
    ax.set_ylim(lo[1] - pad, hi[1] + pad)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (AU)")
    ax.set_ylabel("y (AU)")
    ax.plot([0], [0], "*", ms=14, color="#f59e0b", zorder=5)  # the star sits at a focus

    # Kepler II: slices of equal duration T/n_slices for the highlighted planet, every other one shaded
    T_h = periods[hb] if np.isfinite(periods[hb]) else t[-1]
    edges = np.searchsorted(t, np.arange(n_slices + 1) * T_h / n_slices)
    sectors = []
    for k in range(0, n_slices, 2):
        i0, i1 = edges[k], min(edges[k + 1], len(t) - 1)
        corners = np.vstack([[0.0, 0.0], pos[i0 : i1 + 1, hb]])
        patch = Polygon(corners, closed=True, color=ACCENT[hb % len(ACCENT)], alpha=0.25, lw=0, visible=False)
        ax.add_patch(patch)
        sectors.append((i1, patch))

    lines = [ax.plot([], [], lw=1.4, color=ACCENT[b % len(ACCENT)])[0] for b in range(B)]
    dots = [ax.plot([], [], "o", ms=6, color=ACCENT[b % len(ACCENT)])[0] for b in range(B)]
    ax.set_title(f"shaded sectors: equal Δt = T/{n_slices} → equal areas", fontsize=9)
    clock = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top", fontsize=9)

    # Kepler III: T² against a³ (years and AU, so the slope is exactly 1)
    a3 = a**3
    top = 1.15 * max(a3.max(), np.nanmax(periods**2) if np.isfinite(periods).any() else 1.0)
    ax_k.plot([0, top], [0, top], "--", color="#9ca3af", lw=1, label="T² = a³")
    marks = [
        ax_k.plot([], [], "o", ms=8, color=ACCENT[b % len(ACCENT)], label=f"a={a[b]:.1f}, e={e[b]:.2f}")[0]
        for b in range(B)
    ]
    ax_k.set_xlim(0, top)
    ax_k.set_ylim(0, top)
    ax_k.set_xlabel("a³ (AU³)")
    ax_k.set_ylabel("T² (yr²)  measured")
    ax_k.legend(fontsize=7, loc="upper left")
    ax_k.set_title("third law: dot appears after one orbit", fontsize=9)

    def update(step: int):
        for b in range(B):
            lines[b].set_data(pos[: step + 1, b, 0], pos[: step + 1, b, 1])
            dots[b].set_data([pos[step, b, 0]], [pos[step, b, 1]])
            done = np.isfinite(periods[b]) and t[step] >= periods[b]
            marks[b].set_data([a3[b]] if done else [], [periods[b] ** 2] if done else [])
        for end, patch in sectors:
            patch.set_visible(step >= end)
        clock.set_text(f"t = {t[step]:.2f} yr")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_planets': 4,
    'e_max': 0.6,
    't_max': 3.5,
    'dt': 0.001
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.kepler_orbits")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.kepler_orbits.main")
problem_dir = Path("problems/astro/kepler_orbits")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
